# Tutorial 03

*   Training neural networks in PyTorch
*   Debugging and analyzing overfitting/underfitting

## Neural Networks 

*   Input (features): **x**
*   Output (labels/target values): **y**
*   We want to find a function **f** that will approximate **f(x) = y** and can be run on new data when **y** is unknown

Neural networks approximate the function **f** by composition of functions like

$$ f(x) = f^{(N)}(...f^{(2)}(f^{(1)}(x)))$$

where N is number of layers of the neural network

## Multilayer Perceptron (MLP)

Let's define an ${ f^{(i)}(x) }$ function like

$$ f^{(i)}(x) = \phi(\textbf{W}x + b) $$

where ${x \in \mathbb{R}^{D} }$ is an input vector, ${ \textbf{W} \in \mathbb{R}^{d \times D} }$ is weight matrix, ${ d \in \mathbb{R}^{d} }$ is bias vector to be added, and ${\phi(.)}$ is an activation function.

MLPs are a class of neural network where ${ f_{(i)} }$ is defined like the above. A simple MLP with 2 layers are illustrated in Figure 1

[![MLP Figure 1](https://classic.d2l.ai/_images/mlp.svg)]()
 *Figure 1: A Simple MLP*

 Two of the most commonly used activations are *sigmoid* and *ReLU* functions

 <!-- $$ Sigmoid: \sigma(x) = \frac{1}{1 + e^{-x}} $$
 $$ ReLU: R(x) = max(0, x) $$ -->

  
[![Sigmoid and ReLU figure](https://miro.medium.com/max/1452/1*XxxiA0jJvPrHEJHD4z893g.png )]()



# Download the Dataset
We are using cats vs. dogs dataset used in Tutorial 01. You may download it from [here](https://www.floydhub.com/swaroopgrs/datasets/dogscats) and extract it to your working folder. Alternatively, you can just run the cell below to download and extract it. If you are not running the notebook on Google Colab, make sure that you have gdown by using the command `pip install gdown`. If you are on Windows, the cell won't work. You need to download and extract the data from the link. 

In [ ]:
!gdown --id 1sKjxoH-MQl3CBVEm6C0EHwV3rqtXOqWq
!tar -xf swaroopgrs-datasets-dogscats-1.tar

# Training neural networks
We have a set of input vectors $x$ (features) and desired targets (value / class label) $y$  
We want to find a function $f$ such that $f(x)=y$

Basic training algorithm:  
**Input:**
- features $x$, targets $y$  
- neural network model with randomly initialized parameters $\theta$: $f(x; \theta)$  
- learning_rate $\alpha$  
- loss function $L$  

**Output:**
- optimized parameters $\theta$

**Algorithm**  
>for epoch in 1..n_epochs do
>>     for (minibatch_x, minibatch_y) in dataset do
>>>        compute predicted y' = f(x; theta)
>>>        compute loss = L(y', minibatch_y)
>>>        compute gradients = gradient(L, theta)
>>>        update parameters theta <- theta - learning_rate * gradients

First, set device variable to `cuda` if there is an availabe GPU, otherwise it will be set to `cpu`.

In [ ]:
import torch
from tqdm.notebook import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device: {}'.format(device))

# Dataset
We are defining the dataset object so that Pytorch shall know how to load the dataset, we are using `ImageFolder` class of Pytorch, which puts images in the subfolders of `root_dir` to different classes. Since we have two folder called "cats" and "dogs", we'll have two classes for cats and dogs.

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import Resize, ToTensor, Normalize, Compose
root_dir = 'images'

target_size = (32, 32)
transforms = Compose([Resize(target_size), # Resizes image
                      ToTensor(),           # Converts to Tensor, scales to [0, 1] float (from [0, 255] int)
                      Normalize(mean=(0.5, 0.5, 0.5,), std=(0.5, 0.5, 0.5)), # scales to [-1.0, 1.0]
                    ])

train_dataset_ = ImageFolder('train', transform=transforms)
val_dataset_ = ImageFolder('valid', transform=transforms)

In [ ]:
print('Number of training samples: {}'.format(len(train_dataset_)))
print('Number of validation samples: {}'.format(len(val_dataset_)))

# Load the Dataset
Instead of loading opening image files one by one, we'll load them to memory in the beginning so that the training would be faster.

In [ ]:
class RAMDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, dataset):
        data = []
        for sample in tqdm(dataset):
            data.append(sample)
        self.n = len(dataset)
        self.data = data
        
    def __getitem__(self, ind):
        return self.data[ind]

    def set_transform(self, transform):
        self.transform = transform
    
    def __len__(self):
        return self.n

In [ ]:
train_dataset = RAMDatasetWrapper(train_dataset_)

# Random Sample
Let's pick a random sample from dataset and visualize it with matplotlib

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

index = np.random.randint(0, high=len(train_dataset))
img_to_show = train_dataset[index][0].mul(0.5).add(0.5).permute(1,2,0).cpu().numpy()
label_sample = train_dataset[index][1]
label_dict = {0: 'cat', 1: 'dog'}

plt.imshow(img_to_show)
plt.title('It is a {}'.format(label_dict[label_sample]))

# Dataloaders
Defining the dataloaders for training and validation datasets. This object will do loading of the dataset with desired batch size and shuffles the data.

In [ ]:
from torch.utils.data import DataLoader
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2) #num_workers = n - how many threads in background for efficient loading

In [ ]:
# Same for validation dataset
val_dataset = RAMDatasetWrapper(val_dataset_)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Define the Models
We are declearing the one linear model and one MLP model that we are going to use through this tutorial by inhereting the Pytorch's `nn.Module` class. The input images are converted into 1D vectors in both models, then a binary prediction is made to conclude that the image is cat or dog 

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module):
    def __init__(self, input_dim):
        super(LinearModel, self).__init__()
        self.fc = nn.Linear(input_dim, 2, bias=True) # outputs 2 values - score for cat and score for dog
        
    def forward(self, input):
        out = input.view(input.size(0), -1) # convert batch_size x 3 x imH x imW to batch_size x (3*imH*imW)
        out = self.fc(out) # Applies out = input * A + b. A, b are parameters of nn.Linear that we want to learn
        return out
    
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(MLPModel, self).__init__()
        pass # IMPLEMENT MLP MODEL
    
    def forward(self, input):
        pass # IMPLEMENT the Forward pass

# Training Functions
We are declaring the functions that we'll use for training. `train_epoch` function performs the the training for one epoch, `train` function trains the model for the desired number of epochs. 

In [ ]:
import numpy as np

def train_epoch(model, train_dataloader, optimizer, loss_fn):
    losses = []
    correct_predictions = 0
    # Iterate mini batches over training dataset
    for images, labels in tqdm(train_dataloader):
        # move tensors to current device
     
         
        # Run predictions
        #output= ...
        
        # Set gradients to zero
        # https://pytorch.org/docs/stable/optim.html
        # https://discuss.pytorch.org/t/why-do-we-need-to-set-the-gradients-manually-to-zero-in-pytorch/4903/20
        
        
        # Compute loss
        #loss= ...
        
        # Backpropagate (compute gradients)
        
        
        # Make an optimization step (update parameters)
        
        
        # Log metrics
        losses.append(loss.item())
        predicted_labels = output.argmax(dim=1)
        correct_predictions += (predicted_labels == labels).sum().item()
    accuracy = 100.0 * correct_predictions / len(train_dataloader.dataset)
    # Return loss values for each iteration and accuracy
    mean_loss = np.array(losses).mean()
    return mean_loss, accuracy

In [ ]:
def evaluate(model, dataloader, loss_fn):
    losses = []
    correct_predictions = 0
    with torch.no_grad():
        for images, labels in dataloader:
            
            #move tensors to current device
            
            
            # Run predictions
            #output = ...
            
            
            # Compute loss
            #loss = ...
            
            
            predicted_labels = output.argmax(dim=1)
            correct_predictions += (predicted_labels == labels).sum().item()
            losses.append(loss.item())
    mean_loss = np.array(losses).mean()
    accuracy = 100.0 * correct_predictions / len(dataloader.dataset)
    # Return mean loss and accuracy
    return mean_loss, accuracy

In [ ]:
def train(model, train_dataloader, val_dataloader, optimizer, n_epochs, loss_function):
    # We will monitor loss functions as the training progresses
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(n_epochs):
        # set model to train mode
        
        # train the model for one epoch
        #train_loss, train_accuracy = ...
        
        # set the model to eval model
        
        # evaluate the model with validation data
        #val_loss, val_accuracy = ...
        
        # append losses and accuracies
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print('Epoch {}/{}: train_loss: {:.4f}, train_accuracy: {:.4f}, val_loss: {:.4f}, val_accuracy: {:.4f}'.format(epoch+1, n_epochs,
                                                                                                      train_losses[-1],
                                                                                                      train_accuracies[-1],
                                                                                                      val_losses[-1],
                                                                                                      val_accuracies[-1]))
    return train_losses, val_losses, train_accuracies, val_accuracies

# Plotting
We are defining the plot function to plot the loss vs. epoch and accuracy vs. epoch graphs. The loss and accuracy for training and validation dataset are plotted on the same graph. 

In [ ]:
def plot(n_epochs, train_losses, val_losses, train_accuracies, val_accuracies):
    plt.figure()
    plt.plot(np.arange(n_epochs), train_losses)
    plt.plot(np.arange(n_epochs), val_losses)
    plt.legend(['train_loss', 'val_loss'])
    plt.xlabel('epoch')
    plt.ylabel('loss value')
    plt.title('Train/val loss')

    plt.figure()
    plt.plot(np.arange(n_epochs), train_accuracies)
    plt.plot(np.arange(n_epochs), val_accuracies)
    plt.legend(['train_acc', 'val_acc'])
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.title('Train/val accuracy')


# Linear Model training


In [ ]:
model = LinearModel(32*32*3)
model = model.to(device)

# Train the linear model

In [ ]:
# Visualize losses, accuracies

# MLP training


In [ ]:
model = MLPModel(32*32*3, 64)
model = model.to(device)

# Train the MLP model


In [ ]:
# Visualize losses, accuracies